# Assignment 10: Audio Classification with Speech-to-Text (STT) and Text-to-Speech (TTS)

**Objective:** Design and implement a deep learning audio-classification application that includes STT and TTS, and evaluate it using suitable metrics.

**Application:** Spoken-command classifier and voice assistant  
**Audio dataset:** Google Speech Commands (subset downloaded through TensorFlow Datasets)  
**Audio classifier:** CNN operating on log-Mel spectrograms  
**STT:** SpeechRecognition with Google Web Speech API (requires internet)  
**TTS:** `pyttsx3` offline speech synthesis  
**Evaluation:** Accuracy, macro precision, macro recall, macro F1-score, confusion matrix, and inference time.

> Run the notebook from top to bottom. Dataset download and STT require internet. The TTS output requires a working audio device. If running in Colab, TTS playback may not work directly; save audio or run the application locally for the best demonstration.

## 1. Introduction

Audio classification assigns an audio clip to a predefined category. In this project, the application recognizes spoken commands such as **yes, no, up, down, left, right, on, off, stop, and go**.

The system has three main components:

1. **Speech-to-Text (STT):** Converts a spoken utterance into text.
2. **Audio classification:** A CNN classifies the audio from its log-Mel spectrogram. This provides a separate audio-based prediction that can be compared with the recognized words.
3. **Text-to-Speech (TTS):** Speaks the predicted class and a short response back to the user.

A log-Mel spectrogram represents how audio energy changes over time and frequency. A CNN can learn useful patterns from this two-dimensional representation.

## 2. Workflow

1. Load labelled speech audio.
2. Convert waveforms to a fixed sample rate and duration.
3. Extract log-Mel spectrograms.
4. Train a CNN classifier.
5. Evaluate the model on held-out test data.
6. Record or provide a short speech clip.
7. Run STT and the trained audio classifier.
8. Use TTS to speak the recognized text and model prediction.

## 3. Install and Import Libraries

In [ ]:
# Run this cell if the libraries are not already installed.
%pip install -q tensorflow tensorflow-datasets librosa scikit-learn matplotlib seaborn pandas SpeechRecognition pyttsx3 soundfile

In [ ]:
import os
import time
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
import librosa
import librosa.display

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

## 4. Load the Speech Commands Dataset

In [ ]:
# Speech Commands contains short labelled utterances.
# We use 10 classes to keep this student assignment manageable.
SELECTED_WORDS = [
    "yes", "no", "up", "down", "left",
    "right", "on", "off", "stop", "go"
]
SAMPLE_RATE = 16000
DURATION = 1.0
NUM_SAMPLES = int(SAMPLE_RATE * DURATION)
N_MELS = 64
BATCH_SIZE = 32
EPOCHS = 8

(ds_train, ds_val, ds_test), info = tfds.load(
    "speech_commands",
    split=["train", "validation", "test"],
    as_supervised=True,
    with_info=True
)
print(info)

### Filter the selected words and convert each waveform to a consistent format

In [ ]:
word_to_id = {word: i for i, word in enumerate(SELECTED_WORDS)}
label_names = SELECTED_WORDS

def prepare_example(audio, label):
    # TensorFlow Datasets provides audio as a waveform and label as a string.
    audio = tf.cast(audio, tf.float32)
    audio = tf.reshape(audio, [-1])
    audio = audio[:NUM_SAMPLES]
    padding = tf.maximum(0, NUM_SAMPLES - tf.shape(audio)[0])
    audio = tf.pad(audio, [[0, padding]])
    return audio, label

def collect_split(ds, max_per_class=None):
    waves, labels = [], []
    counts = {word: 0 for word in SELECTED_WORDS}
    for audio, label in tfds.as_numpy(ds):
        word = label.decode("utf-8") if isinstance(label, bytes) else str(label)
        if word not in word_to_id:
            continue
        if max_per_class is not None and counts[word] >= max_per_class:
            continue
        audio = np.asarray(audio, dtype=np.float32).reshape(-1)
        audio = audio[:NUM_SAMPLES]
        if len(audio) < NUM_SAMPLES:
            audio = np.pad(audio, (0, NUM_SAMPLES - len(audio)))
        waves.append(audio)
        labels.append(word_to_id[word])
        counts[word] += 1
    return np.asarray(waves, dtype=np.float32), np.asarray(labels, dtype=np.int64), counts

# Limits can be lowered if memory or runtime is constrained.
X_train_wave, y_train, train_counts = collect_split(ds_train, max_per_class=800)
X_val_wave, y_val, val_counts = collect_split(ds_val, max_per_class=150)
X_test_wave, y_test, test_counts = collect_split(ds_test, max_per_class=150)

print("Train:", X_train_wave.shape, "Validation:", X_val_wave.shape, "Test:", X_test_wave.shape)
print("Training counts:", train_counts)

## 5. Convert Audio to Log-Mel Spectrograms

In [ ]:
def waveform_to_logmel(waveform):
    mel = librosa.feature.melspectrogram(
        y=waveform.astype(np.float32),
        sr=SAMPLE_RATE,
        n_fft=400,
        hop_length=160,
        n_mels=N_MELS,
        fmin=20,
        fmax=7600
    )
    logmel = librosa.power_to_db(mel, ref=np.max)
    # Normalize each spectrogram for stable training.
    logmel = (logmel - logmel.mean()) / (logmel.std() + 1e-6)
    return logmel.astype(np.float32)

X_train = np.stack([waveform_to_logmel(w) for w in X_train_wave])[..., np.newaxis]
X_val = np.stack([waveform_to_logmel(w) for w in X_val_wave])[..., np.newaxis]
X_test = np.stack([waveform_to_logmel(w) for w in X_test_wave])[..., np.newaxis]

print("Log-Mel input shape:", X_train.shape)

### Visualize a sample spectrogram

In [ ]:
sample_index = 0
plt.figure(figsize=(10, 4))
librosa.display.specshow(
    X_train[sample_index, :, :, 0],
    sr=SAMPLE_RATE, hop_length=160, x_axis="time", y_axis="mel"
)
plt.colorbar(format="%+2.0f")
plt.title(f"Log-Mel Spectrogram — {label_names[y_train[sample_index]]}")
plt.tight_layout()
plt.savefig("sample_logmel_spectrogram.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Build the CNN Audio Classifier

The CNN uses convolution layers to learn local time-frequency patterns from the log-Mel spectrogram. Batch normalization and dropout help stabilize training and reduce overfitting. The final softmax layer outputs a probability for each selected word.

In [ ]:
num_classes = len(SELECTED_WORDS)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=X_train.shape[1:]),
    tf.keras.layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

## 7. Train the CNN

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=2, restore_best_weights=True
)

start_train = time.time()
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)
training_seconds = time.time() - start_train
print(f"Training time: {training_seconds:.2f} seconds")

## 8. Plot Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="Train loss")
axes[0].plot(history.history["val_loss"], label="Validation loss")
axes[0].set_title("Training and Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history["accuracy"], label="Train accuracy")
axes[1].plot(history.history["val_accuracy"], label="Validation accuracy")
axes[1].set_title("Training and Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Evaluate the Audio Classifier

In [ ]:
start_inference = time.time()
probabilities = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
inference_seconds = time.time() - start_inference

y_pred = probabilities.argmax(axis=1)
test_accuracy = accuracy_score(y_test, y_pred)
macro_precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
macro_recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

metrics_df = pd.DataFrame([{
    "Test accuracy": test_accuracy,
    "Macro precision": macro_precision,
    "Macro recall": macro_recall,
    "Macro F1-score": macro_f1,
    "Training time (seconds)": training_seconds,
    "Average inference time per clip (ms)": (inference_seconds / len(X_test)) * 1000
}])
display(metrics_df.round(4))

### Classification report and confusion matrix

In [ ]:
print(classification_report(
    y_test, y_pred, labels=list(range(num_classes)),
    target_names=label_names, zero_division=0
))

cm = confusion_matrix(y_test, y_pred, labels=list(range(num_classes)))
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_names, yticklabels=label_names)
plt.title("CNN Audio Classification — Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Integrate Speech-to-Text (STT)

The following helper uses the SpeechRecognition package and Google's Web Speech API. It requires internet access and a clear audio clip. For reproducibility, provide a WAV file recorded at 16 kHz where possible.

If you are using a local microphone, the notebook attempts to capture speech directly. In Colab, microphone access through Python may not work reliably; use a WAV file instead.

In [ ]:
import speech_recognition as sr

recognizer = sr.Recognizer()

def speech_to_text_from_file(wav_path):
    with sr.AudioFile(wav_path) as source:
        audio_data = recognizer.record(source)
    try:
        text = recognizer.recognize_google(audio_data)
        return text
    except sr.UnknownValueError:
        return "Speech could not be understood."
    except sr.RequestError as exc:
        return f"STT service error: {exc}"

# Example after you upload/record a WAV file:
# recognized_text = speech_to_text_from_file("my_command.wav")
# print("Recognized text:", recognized_text)

## 11. Predict the Audio Class from a WAV File

The CNN takes the audio waveform as input, independently of STT. It resamples the file to 16 kHz, pads/truncates it to one second, generates a log-Mel spectrogram, and predicts one of the 10 dataset classes.

In [ ]:
def classify_wav_file(wav_path):
    waveform, sr_original = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True, duration=DURATION)
    if len(waveform) < NUM_SAMPLES:
        waveform = np.pad(waveform, (0, NUM_SAMPLES - len(waveform)))
    else:
        waveform = waveform[:NUM_SAMPLES]

    features = waveform_to_logmel(waveform)[np.newaxis, ..., np.newaxis]
    start = time.time()
    probs = model.predict(features, verbose=0)[0]
    elapsed_ms = (time.time() - start) * 1000
    predicted_id = int(np.argmax(probs))
    return label_names[predicted_id], float(probs[predicted_id]), elapsed_ms

# Example:
# predicted_word, confidence, latency_ms = classify_wav_file("my_command.wav")
# print(predicted_word, confidence, latency_ms)

## 12. Integrate Text-to-Speech (TTS)

`pyttsx3` uses the local system's speech engine and can work offline on a supported desktop. On Windows it generally uses SAPI5. Audio playback may not work inside hosted notebook environments.

In [ ]:
import pyttsx3

def speak_text(text):
    try:
        engine = pyttsx3.init()
        engine.setProperty("rate", 165)
        engine.say(text)
        engine.runAndWait()
        engine.stop()
    except Exception as exc:
        print("TTS could not play audio in this environment:", exc)

# Example:
# speak_text("Hello! Your command was recognized.")

## 13. Complete Voice Assistant Demonstration

Put a short WAV recording of one of the selected words into the working directory, and set `AUDIO_FILE` to its path. STT may recognize a word or phrase, while the CNN predicts one of the 10 target classes from the audio itself. These outputs can differ because STT and classification use different models and objectives.

In [ ]:
AUDIO_FILE = "my_command.wav"  # Replace with your WAV filename.

if os.path.exists(AUDIO_FILE):
    recognized_text = speech_to_text_from_file(AUDIO_FILE)
    predicted_word, confidence, latency_ms = classify_wav_file(AUDIO_FILE)

    print("STT transcript:", recognized_text)
    print("CNN predicted command:", predicted_word)
    print(f"CNN confidence: {confidence:.3f}")
    print(f"CNN inference time: {latency_ms:.2f} ms")

    response = (
        f"I heard: {recognized_text}. "
        f"The audio classifier predicts: {predicted_word}."
    )
    speak_text(response)
else:
    print(f"Add a WAV recording named '{AUDIO_FILE}' to run the end-to-end demonstration.")
    print("The training and evaluation sections can still be completed without a personal recording.")

## 14. Save the Model and Evaluation Results

In [ ]:
model.save("audio_command_cnn.keras")
metrics_df.to_csv("evaluation_metrics.csv", index=False)
print("Saved audio_command_cnn.keras and evaluation_metrics.csv")

## 15. Results and Discussion

Use the actual metrics printed in Section 9 and the classification report to discuss:

1. **Accuracy:** The proportion of test clips assigned the correct word.
2. **Macro precision:** Average precision across the selected classes, giving each class equal weight.
3. **Macro recall:** Average recall across the classes.
4. **Macro F1-score:** Harmonic mean of precision and recall for each class, averaged across classes.
5. **Inference latency:** Approximate time per test clip in the current environment; it depends on hardware and batch size.
6. **Confusion matrix:** Shows which spoken commands are commonly confused.
7. **STT quality:** Manually inspect the transcript on several clear recordings. A formal STT evaluation should use reference transcripts and Word Error Rate (WER); this notebook does not claim a WER score without ground-truth transcripts for the supplied recordings.
8. **TTS quality:** Confirm that the spoken response is intelligible and plays successfully on the target device. TTS naturalness needs a human listening evaluation or a suitable perceptual metric.

### Limitations
- The CNN is trained on a selected subset of 10 Speech Commands classes, not arbitrary conversational speech.
- Google STT requires internet and may have service or privacy limitations.
- `pyttsx3` depends on local system speech engines and audio devices.
- The audio classifier and STT can disagree because they perform different tasks.

## 16. Conclusion

This assignment implements a spoken-command application that combines a CNN-based audio classifier with STT and TTS. Audio waveforms are transformed into log-Mel spectrograms, and the CNN learns to classify ten spoken commands. The classifier is evaluated using test accuracy, macro precision, macro recall, macro F1-score, inference time, and a confusion matrix. STT converts speech into text, while TTS speaks the transcript and the classifier's prediction.

The final report should include the metrics and graphs produced by your own run. Performance varies with dataset subset, training duration, environment, microphone quality, and the speech recognition service.

## 17. References

1. Warden, P. “Speech Commands: A Dataset for Limited-Vocabulary Speech Recognition,” 2018. https://arxiv.org/abs/1804.03209
2. TensorFlow Datasets — Speech Commands: https://www.tensorflow.org/datasets/catalog/speech_commands
3. TensorFlow/Keras documentation: https://www.tensorflow.org/guide/keras
4. Librosa documentation: https://librosa.org/doc/latest/index.html
5. SpeechRecognition package: https://pypi.org/project/SpeechRecognition/
6. pyttsx3 documentation: https://pyttsx3.readthedocs.io/